# 01 — Data Exploration

AppleSupport subset exploration and reproducible customer-message extraction.

In [ ]:
from pathlib import Path
import pandas as pd, re
ROOT=Path('..')
DATA=ROOT/'data'
if not DATA.exists(): DATA=Path('data')
path=DATA/'applesupport_conversations.csv'
if not path.exists():
    raise FileNotFoundError('Place applesupport_conversations.csv in data/ before running this notebook.')
df=pd.read_csv(path).fillna('')
print(df.shape)
display(df.head())


In [ ]:
def extract_pairs(conversation):
    return re.findall(r'Customer:\s*(.*?)\s*Support:\s*(.*?)(?=Customer:|$)', str(conversation), flags=re.S|re.I)
rows=[]
for _,r in df.iterrows():
    for c,s in extract_pairs(r['conversation']):
        rows.append({'conversation_id':r.get('conversation_id',''),'clean_message':re.sub(r'\s+',' ',c).strip(),'support_reply':re.sub(r'\s+',' ',s).strip()})
history=pd.DataFrame(rows).drop_duplicates().reset_index(drop=True)
print('Historical pairs:',history.shape)
display(history.head())
history.to_csv(DATA/'historical_support_pairs.csv',index=False)


In [ ]:
print('Conversation count:',df['conversation_id'].nunique() if 'conversation_id' in df else len(df))
print('Historical pairs:',len(history))
print('Median customer message length:',history.clean_message.str.len().median())
